In [2]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 855, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 855 (delta 24), reused 27 (delta 9), pack-reused 791 (from 1)
Receiving objects: 100% (855/855), 5.85 MiB | 30.12 MiB/s, done.
Resolving deltas: 100% (284/284), done.
Mounted at /content/drive


In [3]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00


In [4]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [5]:
from utils.datasets   import NF_IDS_Dataset
from utils.models     import E_GraphSAGE
from utils.metrics    import calculate_metrics_gnn
from utils.training   import train_epoch, evaluate, find_optimal_threshold, set_seed, run_multiple_seeds
from utils.experiment import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots

# Auxiliary

In [6]:
ROOT_PATH = "./dataset_processed"

In [19]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [8]:
ROOT_DIR = "./results_earlystopping"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")


# Main

## Seeds

In [9]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # vector 1s
EDGE_DIM = 32   # 7 dst_port + 5 protocol + 20 numeric
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [10]:
model_config = {
    "model_name": None,
    "type": "E_GraphSAGE",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE,
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [11]:
EXPERIMENT_NAME="EGraphSAGE_BiasOn"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [24]:
run_multiple_seeds(
    model_class=E_GraphSAGE,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: EGraphSAGE_BiasOn
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=EGraphSAGE_BiasOn_seed42_20260421_181148

EGraphSAGE_BiasOn_seed42
   Ep 1 | Loss: 0.2210 | Val Loss: 0.2371 | Val AUC-PR: 0.0669 (*)
   Ep 2 | Loss: 0.2188 | Val Loss: 0.2321 | Val AUC-PR: 0.0767 (*)
   Ep 3 | Loss: 0.2062 | Val Loss: 0.2333 | Val AUC-PR: 0.1034 (*)
   Ep 5 | Loss: 0.1949 | Val Loss: 0.2283 | Val AUC-PR: 0.1303 (*)
   Ep 7 | Loss: 0.1900 | Val Loss: 0.2256 | Val AUC-PR: 0.1747 (*)
   Ep 8 | Loss: 0.1952 | Val Loss: 0.2285 | Val AUC-PR: 0.2098 (*)
   Ep 9 | Loss: 0.1977 | Val Loss: 0.2149 | Val AUC-PR: 0.2525 (*)
   Ep 10 | Loss: 0.1935 | Val Loss: 0.2167 | Val AUC-PR: 0.2850 (*)
   Ep 11 | Loss: 0.1949 | Val Loss: 0.2250 | Val AUC-PR: 0.2908 (*)
   Ep 13 | Loss: 0.1853 | Val Loss: 0.2225 | Val AUC-PR: 0.3054 (*)
   Ep 14 | Loss: 0.1905 | Val Loss: 0.2085 | Val AUC-PR: 0.3075 (*)
   Ep 15 | Loss: 0.1833 | 